# Train flush CNN for Arduino Nano 33 BLE Sense

This notebook is **self-contained**. Upload **`dataset_toiletflush_16k.zip`**.

Zip the folder so it contains:

- `positives/*.wav` = flush (label 1)
- `negatives/*.wav` = not flush (label 0)

Clips should be **16 kHz mono**, about **8 s**. Other rates are resampled. Short files are zero-padded; long files are cropped to 8 s.

Output: **250 × 32** log-mel (32 ms hop, 64 ms FFT) → int8 TFLite → `model.h` for `FlushFan/`.

Copy `model.h` over `FlushFan/model.h`, then flash `FlushFan.ino`. GPU is optional.

In [ ]:
%pip install -q soundfile soxr numpy
import tensorflow as tf
print("TensorFlow", tf.__version__)

## Upload dataset zip

Upload **`dataset_toiletflush_16k.zip`**. Inside it should be `positives/` and `negatives/` (zip the `dataset_toiletflush_16k` folder, not a flat list of wavs).

Set `USE_DRIVE = True` if that folder is already on Google Drive.

In [ ]:
from pathlib import Path
from google.colab import files, drive
import zipfile, os

USE_DRIVE = False
DRIVE_ROOT = Path("/content/drive/MyDrive/dataset_toiletflush_16k")


def extract_zips(uploaded):
    for name in uploaded:
        if str(name).endswith(".zip"):
            print("extracting", name)
            with zipfile.ZipFile(name) as z:
                z.extractall("/content")


def wav_count(folder):
    return len(list(folder.glob("*.wav"))) if folder.is_dir() else 0


def negative_dir(root):
    for name in ("negatives", "tails"):
        d = root / name
        if wav_count(d):
            return d
    return None


def find_dataset():
    hits = []
    for p in Path("/content").rglob("*"):
        if not p.is_dir():
            continue
        if wav_count(p / "positives") and negative_dir(p) is not None:
            hits.append(p)
    if not hits:
        return None
    named = [p for p in hits if p.name == "dataset_toiletflush_16k"]
    return (named or hits)[0]


os.chdir("/content")
if USE_DRIVE:
    drive.mount("/content/drive")
    data = DRIVE_ROOT if wav_count(DRIVE_ROOT / "positives") else find_dataset()
else:
    print("Upload dataset_toiletflush_16k.zip (positives/ and negatives/ inside)")
    extract_zips(files.upload())
    data = find_dataset()

assert data is not None, (
    "Need positives/*.wav and negatives/*.wav. "
    "Zip the dataset_toiletflush_16k folder."
)
neg = negative_dir(data)
assert neg is not None
print("dataset:", data)
print("  positives", wav_count(data / "positives"))
print(f"  {neg.name:10}", wav_count(neg))

## MFE (must match Arduino `mfe.cpp`)

In [ ]:
import numpy as np
import soxr
import soundfile as sf

SAMPLE_RATE = 16000
WINDOW_S = 8.0
N_FFT = 1024
HOP = 512
N_MELS = 32
N_FRAMES = 250
SAMPLES_PER_CLIP = SAMPLE_RATE * 8
STFT_SAMPLES = (N_FRAMES - 1) * HOP + N_FFT
EPS = 1e-6


def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + np.asarray(hz) / 700.0)


def mel_to_hz(mel):
    return 700.0 * (10.0 ** (np.asarray(mel) / 2595.0) - 1.0)


def hann_window(n=N_FFT):
    return (0.5 - 0.5 * np.cos(2.0 * np.pi * np.arange(n, dtype=np.float64) / n)).astype(np.float32)


def mel_filterbank(n_mels=N_MELS, n_fft=N_FFT, sr=SAMPLE_RATE, fmin=0.0, fmax=8000.0):
    n_bins = n_fft // 2 + 1
    mels = np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_mels + 2)
    hz = mel_to_hz(mels)
    bins = np.floor((n_fft + 1) * hz / sr).astype(int)
    bins = np.clip(bins, 0, n_bins - 1)
    fb = np.zeros((n_mels, n_bins), dtype=np.float32)
    for i in range(n_mels):
        left, center, right = bins[i], bins[i + 1], bins[i + 2]
        if center == left:
            center = min(left + 1, n_bins - 1)
        if right == center:
            right = min(center + 1, n_bins - 1)
        for j in range(left, center):
            fb[i, j] = (j - left) / (center - left)
        for j in range(center, right):
            fb[i, j] = (right - j) / (right - center)
        s = fb[i].sum()
        if s > 0:
            fb[i] /= s
    return fb


WINDOW = hann_window()
FILTERBANK = mel_filterbank()


def mfe_frames(pcm):
    x = np.asarray(pcm, dtype=np.float32).reshape(-1)
    if len(x) < N_FFT:
        x = np.pad(x, (0, N_FFT - len(x)))
    n_frames = 1 + (len(x) - N_FFT) // HOP
    windows = np.lib.stride_tricks.sliding_window_view(x, N_FFT)[::HOP][:n_frames]
    frames = windows * WINDOW
    spec = np.fft.rfft(frames, n=N_FFT, axis=1)
    power = (spec.real * spec.real + spec.imag * spec.imag).astype(np.float32)
    mel = power @ FILTERBANK.T
    return np.log(mel + EPS).astype(np.float32)


def mfe_clip(pcm):
    x = np.asarray(pcm, dtype=np.float32).reshape(-1)
    if len(x) < SAMPLES_PER_CLIP:
        x = np.pad(x, (0, SAMPLES_PER_CLIP - len(x)))
    else:
        x = x[:SAMPLES_PER_CLIP]
    if len(x) < STFT_SAMPLES:
        x = np.pad(x, (0, STFT_SAMPLES - len(x)))
    spec = mfe_frames(x)
    if spec.shape[0] < N_FRAMES:
        spec = np.pad(spec, ((0, N_FRAMES - spec.shape[0]), (0, 0)))
    return spec[:N_FRAMES]


def load_wavs(folder, label):
    files = sorted(folder.rglob("*.wav"))
    if not files:
        raise FileNotFoundError(f"No wav files in {folder}")
    specs, labels = [], []
    for i, path in enumerate(files, 1):
        pcm, sr = sf.read(str(path), dtype="float32", always_2d=False)
        if pcm.ndim > 1:
            pcm = pcm.mean(axis=1)
        pcm = np.asarray(pcm, dtype=np.float32).reshape(-1)
        if sr != SAMPLE_RATE:
            pcm = soxr.resample(pcm, sr, SAMPLE_RATE, quality="VHQ")
        specs.append(mfe_clip(pcm))
        labels.append(label)
        if i % 50 == 0 or i == len(files):
            print(f"  {folder.name}: {i}/{len(files)}")
    x = np.stack(specs).astype(np.float32)[..., np.newaxis]
    y = np.array(labels, dtype=np.int32)
    return x, y

print("MFE ready:", N_FRAMES, "x", N_MELS)

## Preview training spectrograms

Same **250 × 32** log-mel as Arduino / training (64 ms FFT, 32 ms hop). Time →, low mel at the bottom. Change `N_SHOW` to plot more clips. Run after the dataset and MFE cells.

In [ ]:
import matplotlib.pyplot as plt

assert "data" in dir() and data is not None, "Run the upload cell first."
assert "mfe_clip" in dir(), "Run the MFE cell first."

N_SHOW = 4
rng = np.random.default_rng(0)


def load_spec(path):
    pcm, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if pcm.ndim > 1:
        pcm = pcm.mean(axis=1)
    pcm = np.asarray(pcm, dtype=np.float32).reshape(-1)
    if sr != SAMPLE_RATE:
        pcm = soxr.resample(pcm, sr, SAMPLE_RATE, quality="VHQ")
    return mfe_clip(pcm)


def pick(folder):
    files = sorted(folder.rglob("*.wav"))
    n = min(N_SHOW, len(files))
    if n == 0:
        raise FileNotFoundError(f"No wavs in {folder}")
    ix = rng.choice(len(files), size=n, replace=False)
    return [files[int(i)] for i in np.atleast_1d(ix)]


pos_files = pick(data / "positives")
neg_files = pick(neg)
n_cols = max(len(pos_files), len(neg_files))
fig, axes = plt.subplots(2, n_cols, figsize=(3.4 * n_cols, 6.2), squeeze=False)
fig.suptitle("Training MFE (250 frames × 32 mels, ~8 s)", fontsize=12)

for row, (files, label) in enumerate(((pos_files, "flush"), (neg_files, "not flush"))):
    for col in range(n_cols):
        ax = axes[row][col]
        if col >= len(files):
            ax.axis("off")
            continue
        spec = load_spec(files[col])
        ax.imshow(
            spec.T,
            origin="lower",
            aspect="auto",
            extent=[0, WINDOW_S, 0, N_MELS],
            cmap="magma",
        )
        ax.set_title(f"{label}: {files[col].name}", fontsize=8)
        ax.set_xlabel("time (s)")
        if col == 0:
            ax.set_ylabel("mel bin")

plt.tight_layout()
plt.show()
print("These are the tensors the CNN trains on (before time-roll / gain / noise augment).")


## Train, quantize, download `model.h`

In [ ]:
from google.colab import files
from tensorflow import keras
from tensorflow.keras import layers

EPOCHS = 40
BATCH = 32
SEED = 0
OUT = Path("/content/flush_export")
OUT.mkdir(exist_ok=True)

assert "data" in dir() and data is not None, "Run the upload cell first."

print("Loading", data)
x_pos, y_pos = load_wavs(data / "positives", 1)
x_neg, y_neg = load_wavs(neg, 0)
x = np.concatenate([x_pos, x_neg], axis=0)
y = np.concatenate([y_pos, y_neg], axis=0)
print(f"clips: {len(y)}  positives={int(y.sum())}  negatives={int((1 - y).sum())}")
print(f"MFE shape: {x.shape}  range [{x.min():.2f}, {x.max():.2f}]")
assert x.shape[1:] == (N_FRAMES, N_MELS, 1), x.shape

rng = np.random.default_rng(SEED)
idx = rng.permutation(len(y))
n_val = max(1, int(len(y) * 0.2))
va, tr = idx[:n_val], idx[n_val:]
x_tr, y_tr = x[tr], y[tr]
x_va, y_va = x[va], y[va]


def augment(spec, label):
    spec = tf.roll(spec, tf.random.uniform([], -4, 5, dtype=tf.int32), axis=0)
    spec = spec * tf.random.uniform([], 0.7, 1.3)
    spec = spec + tf.random.normal(tf.shape(spec), stddev=0.05)
    return spec, label


train_ds = (
    tf.data.Dataset.from_tensor_slices((x_tr, y_tr))
    .shuffle(len(x_tr), seed=SEED)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = tf.data.Dataset.from_tensor_slices((x_va, y_va)).batch(BATCH)

inp = layers.Input(shape=(N_FRAMES, N_MELS, 1), name="mfe")
h = layers.Conv2D(8, (3, 3), strides=(2, 2), padding="same", activation="relu")(inp)
h = layers.Conv2D(16, (3, 3), strides=(2, 2), padding="same", activation="relu")(h)
h = layers.Conv2D(16, (3, 3), padding="same", activation="relu")(h)
h = layers.GlobalAveragePooling2D()(h)
h = layers.Dense(16, activation="relu")(h)
out = layers.Dense(2, name="logits")(h)
model = keras.Model(inp, out, name="flush_cnn")
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.summary()
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=8, restore_best_weights=True, mode="max"
        )
    ],
)

print("Quantizing to int8 TFLite ...")


def representative():
    n = min(100, len(x_tr))
    for i in range(n):
        yield [x_tr[i : i + 1]]


converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
blob = converter.convert()
(OUT / "flush_int8.tflite").write_bytes(blob)

lines = [
    "// Auto-generated by colab_train_flush.ipynb — do not edit.",
    "// Copy this file into the FlushFan sketch folder.",
    "#pragma once",
    "",
    f"constexpr unsigned int g_flush_model_len = {len(blob)};",
    "alignas(8) const unsigned char g_flush_model[] = {",
]
for i in range(0, len(blob), 12):
    chunk = ", ".join(f"0x{b:02x}" for b in blob[i : i + 12])
    comma = "," if i + 12 < len(blob) else ""
    lines.append(f"  {chunk}{comma}")
lines.append("};\n")
(OUT / "model.h").write_text("\n".join(lines), encoding="utf-8")

interp = tf.lite.Interpreter(model_content=blob)
interp.allocate_tensors()
tinp = interp.get_input_details()[0]
tout = interp.get_output_details()[0]
scale, zp = tinp["quantization"]
correct = 0
for i in range(len(x_va)):
    q = np.round(x_va[i] / scale + zp).astype(np.int8)
    interp.set_tensor(tinp["index"], q[np.newaxis, ...])
    interp.invoke()
    pred = int(np.argmax(interp.get_tensor(tout["index"])[0]))
    correct += int(pred == y_va[i])
print(f"int8 TFLite val accuracy: {correct / len(y_va):.3f}")
print(f"model size: {len(blob)} bytes")
print("Wrote", OUT / "model.h", (OUT / "model.h").stat().st_size)
print("Wrote", OUT / "flush_int8.tflite", (OUT / "flush_int8.tflite").stat().st_size)

files.download(str(OUT / "model.h"))
files.download(str(OUT / "flush_int8.tflite"))
print("Copy model.h over FlushFan/model.h, then upload the sketch.")